# Task 4: PyTorch Deep Neural Network vs. Classical Machine Learning
**Course:** AI in Cyber Security Lab (`ENSP355`) — Unit 1 Capstone  
**Student:** Yatharth Chopra (`2301420022`) | **Program:** B.Tech CSE (Data Science), Sem 7  
**University:** K.R. Mangalam University | **Faculty:** Monika Khatkar

---

### Deep Learning Architecture (`PhishingDNN` in PyTorch)
We build a 3-hidden-layer feed-forward neural network (`27 → 128 → Dropout(0.30) → 64 → Dropout(0.20) → 32 → 1`) trained with `BCEWithLogitsLoss`, `Adam` (`lr=1e-3`, `weight_decay=1e-4`), a 15% validation split, and **early stopping** (`patience=6`). We benchmark it against all four classical ML models on the exact same split and across **5 repeated stratified splits**.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from common import OUT_DIR, RANDOM_STATE, load_data
from preprocessing import split_data

spec = importlib.util.spec_from_file_location("dl_cmp", ROOT / "src" / "04_dl_comparison.py")
dl_cmp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dl_cmp)

df = load_data()
X_train, X_test, y_train, y_test = split_data(df)

## 1) Train PyTorch DNN & Display Epoch-by-Epoch Convergence

In [2]:
dnn_model, dnn_res, dnn_pred, dnn_prob, dnn_hist = dl_cmp.train_pytorch_dnn(
    X_train, X_test, y_train, y_test, seed=RANDOM_STATE
)
for ep, (tl, vl, va) in enumerate(zip(dnn_hist["train_loss"], dnn_hist["val_loss"], dnn_hist["val_acc"]), 1):
    if ep == 1 or ep % 5 == 0 or ep == len(dnn_hist["train_loss"]):
        print(f"Epoch {ep:02d} | Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | Val Acc: {va:.4f}")

print("\nPyTorch DNN Test Metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in dnn_res.items()})

Epoch 01 | Train Loss: 0.5956 | Val Loss: 0.3670 | Val Acc: 0.8972
Epoch 05 | Train Loss: 0.2388 | Val Loss: 0.2157 | Val Acc: 0.9080
Epoch 10 | Train Loss: 0.2125 | Val Loss: 0.1985 | Val Acc: 0.9174
Epoch 15 | Train Loss: 0.2021 | Val Loss: 0.1992 | Val Acc: 0.9174
Epoch 20 | Train Loss: 0.1963 | Val Loss: 0.1871 | Val Acc: 0.9228
Epoch 25 | Train Loss: 0.1909 | Val Loss: 0.1885 | Val Acc: 0.9194
Epoch 30 | Train Loss: 0.1857 | Val Loss: 0.1903 | Val Acc: 0.9208
Epoch 35 | Train Loss: 0.1804 | Val Loss: 0.1869 | Val Acc: 0.9194
Epoch 40 | Train Loss: 0.1797 | Val Loss: 0.1848 | Val Acc: 0.9208

PyTorch DNN Test Metrics: {'accuracy': 0.9259, 'precision': 0.937, 'recall': 0.8992, 'f1': 0.9177, 'roc_auc': 0.9776, 'pr_auc': 0.9766, 'tn': 1272, 'fp': 69, 'fn': 115, 'tp': 1026, 'train_time_s': 4.5658, 'predict_time_ms': 0.8476, 'acc_seen_profiles': 0.9288, 'acc_unseen_profiles': 0.8689, 'share_unseen_profiles': 0.0492}


## 2) Full Benchmark Comparison (Single Split & 5-Split Stability)

In [3]:
cmp_df = pd.read_csv(OUT_DIR / "model_comparison.csv")
print("Single Split Comparison (Classical ML vs PyTorch DNN):")
display(cmp_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "tn", "fp", "fn", "tp", "train_time_s"]].round(4))

rep_df = pd.read_csv(OUT_DIR / "model_comparison_repeated.csv", index_col=0)
print("\n5-Split Repeated Stability (Mean +/- Std):")
display(rep_df.round(4))

Single Split Comparison (Classical ML vs PyTorch DNN):


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp,train_time_s
0,Logistic Regression,0.8985,0.9030,0.8729,0.8877,0.9643,0.9596,1234,107,145,996,0.0410
1,Decision Tree,0.9053,0.9074,0.8843,0.8957,0.9705,0.9638,1238,103,132,1009,0.0377
2,Random Forest,0.9255,0.9322,0.9036,0.9177,0.9797,0.9786,1266,75,110,1031,0.5154
3,Gradient Boosting,0.9255,0.9338,0.9018,0.9175,0.9788,0.9774,1268,73,112,1029,2.0588
4,PyTorch DNN,0.9259,0.9370,0.8992,0.9177,0.9776,0.9766,1272,69,115,1026,8.2413



5-Split Repeated Stability (Mean +/- Std):


,accuracy_mean,accuracy_std,f1_mean,f1_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,acc_unseen_profiles_mean,acc_unseen_profiles_std,train_time_s_mean,train_time_s_std
model,,,,,,,,,,,,,,
Logistic Regression,0.9001,0.0021,0.8893,0.0027,0.8729,0.0069,0.9619,0.0027,0.9551,0.0056,0.7363,0.0567,0.0296,0.0065
Decision Tree,0.9060,0.0052,0.8968,0.0062,0.8883,0.0115,0.9695,0.0019,0.9599,0.0035,0.8141,0.0387,0.0295,0.0100
Random Forest,0.9242,0.0027,0.9161,0.0032,0.9008,0.0059,0.9785,0.0016,0.9770,0.0021,0.8648,0.0306,0.4529,0.1589
Gradient Boosting,0.9251,0.0030,0.9171,0.0034,0.9022,0.0047,0.9784,0.0019,0.9767,0.0024,0.8631,0.0266,0.6682,0.0543
PyTorch DNN,0.9218,0.0045,0.9144,0.0042,0.9074,0.0110,0.9768,0.0023,0.9750,0.0025,0.8543,0.0306,6.1877,2.0328


## 3) ROC Curves & PyTorch Training/Validation Loss Curve

In [4]:
from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
axes[0].imshow(Image.open(ROOT / "outputs" / "plots" / "13_roc_curves.png"))
axes[0].axis("off")
axes[1].imshow(Image.open(ROOT / "outputs" / "plots" / "11_mlp_learning_curve.png"))
axes[1].axis("off")
plt.tight_layout()
plt.show()

C:\Users\yatha\AppData\Local\Temp\ipykernel_13204\1115711942.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
